# Market Data Quality Framework

Module: Markets and Data

## Lesson summary

Financial data analysis starts before the first model is fit. Market data must be interpreted through the trading venue, instrument type, calendar, corporate action policy, source limitations, and data-cleaning assumptions that produced it {cite}`fabozzi2019foundations,tsay2010analysis`.

This lesson synthesizes the data-quality part of Module 1 into a practical framework for actuarial and quantitative finance work. A structured table can still contain serious errors: missing dates, stale prices, inconsistent calendars, duplicated records, incorrect units, unadjusted corporate actions, or undocumented macro revisions.

## Learning objectives

By the end of this lesson, students should be able to:

- identify the main institutions and data sources in the Mexican financial market;
- distinguish price, volume, macroeconomic, fixed-income, FX, and derivative data;
- build a source inventory before modeling;
- distinguish missing dates from missing values;
- detect missing data, asynchronous calendars, frequency mismatches, outliers, stale prices, revisions, and corporate action problems;
- transform adjusted prices into simple and log returns;
- document data assumptions clearly enough for a risk, actuarial, or reproducibility review.

## Setup

Use the deterministic classroom examples and helper functions from `src.market_data_quality`; no external provider call is required for the default run.

## Mexican market structure

| Institution or venue | Role in the data workflow | Typical course use |
| --- | --- | --- |
| Banxico | Central bank, monetary policy, exchange rates, rates, macro-financial series | Target rate, TIIE, CETES, FIX, UDI {cite}`banxicoSIE2025` |
| CNBV | Financial-sector supervision and regulatory reporting | Institutional context and counterparty risk |
| BMV and BIVA | Equity and listed securities venues | Mexican equity prices and index context |
| MexDer | Listed derivatives venue | FX, TIIE, and equity-index hedging examples |
| PIP and Valmer | Independent price vendors | Valuation context for less liquid instruments |

The key modeling lesson is that prices are not abstract numbers. They are produced by institutions, market conventions, trading calendars, and quotation rules.

## Data source inventory

Every dataset used in the course should be documented before modeling:

| Field | Example |
| --- | --- |
| Provider | Banxico, FRED, Yahoo Finance, exchange file, instructor sample {cite}`banxicoSIE2025,fredAPI2025,yfinance2025` |
| Instrument or variable | `^MXX`, USD/MXN FIX, 28-day TIIE, CPI |
| Frequency | daily, weekly, monthly, intraday |
| Date range | first and last available observation |
| Price field | close, adjusted close, settlement price, bid, ask, mid |
| Currency | MXN, USD, UDI, real terms |
| Calendar | local market calendar, US business calendar, merged business calendar |
| Known limitations | missing values, rate limits, unofficial endpoints, revisions |

## Quality problem map

Structure is not the same as quality. The most common problems in financial and macroeconomic datasets are:

| Problem | Example |
| --- | --- |
| Missing dates | A market holiday, API outage, or absent observation |
| Missing values | A price field is blank for one instrument |
| Duplicates | The same date and ticker appear more than once |
| Calendar mismatch | U.S. and Mexican markets have different holidays |
| Frequency mismatch | Daily prices are merged with monthly inflation |
| Unit mismatch | Percent, decimal, index level, pesos, dollars, and basis points are mixed |
| Stale prices | An illiquid asset repeats the same value for many days |
| Outliers | A return appears extremely large because of a true event or a data error |
| Corporate actions | Splits or dividends are not reflected correctly |
| Revisions | Macroeconomic values change after initial publication |
| Survivorship bias | Only currently listed instruments are included |
| Look-ahead bias | Future information is accidentally used in a historical decision |

The purpose is not to solve every advanced bias immediately. The purpose is to train students to suspect the data before trusting the result.

## Example Banxico series

These series are useful for future Mexican market notebooks. Verify availability before live classroom use.

| Series | Meaning | Possible use |
| --- | --- | --- |
| `SF61745` | Target rate | monetary policy context |
| `SF60648` | 28-day TIIE | interbank and floating-rate examples |
| `SF60633` | 28-day CETES | short risk-free proxy |
| `SF43718` | USD/MXN FIX | FX risk and macro dashboards |
| `SP68257` | UDI | inflation-linked valuation context |

## Data quality pipeline

1. Define the instrument universe and provider.
2. Download raw data and preserve the original field names.
3. Build a data source inventory.
4. Normalize the index to a `DatetimeIndex`.
5. Align calendars only after understanding market holidays.
6. Use adjusted prices for equity return calculations.
7. Convert prices to simple or log returns.
8. Audit missingness and outliers.
9. Document every cleaning assumption.
10. Pass only cleaned data into models.

## Missing dates versus missing values

A **missing date** means an expected date is absent from the index. A **missing value** means the date exists, but one or more fields are empty.

This distinction matters because a missing trading day may be normal if the market was closed, while a missing price on an active trading day may indicate an extraction or data-quality issue. A correct pipeline should distinguish among market closed, data not yet published, provider error, source unavailability, and values that are not applicable to the instrument.

## Calendar and frequency differences

Financial instruments do not all follow the same calendar. Mexican equities, U.S. equities, government securities, exchange rates, and macroeconomic indicators may have different holidays, time zones, and publication schedules.

A typical panel may combine:

- daily stock prices;
- daily exchange rates;
- weekly monetary data;
- monthly inflation;
- monthly industrial activity;
- quarterly GDP;
- annual financial statements.

These series cannot be merged mechanically. The analyst must define the alignment rule. Examples include using month-end values for market variables, assigning macro releases to publication dates instead of reference periods, avoiding forward-fill unless the assumption is explicit, and using only information available at the historical decision date when testing a strategy.

## Returns

Simple return:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}.
$$

Log return:

$$
r_t = \ln(P_t) - \ln(P_{t-1}).
$$

Log returns are additive across time and are the default input for many statistical models of asset returns {cite}`tsay2010analysis`.

In [ ]:
import pandas as pd

from src.market_data_quality import data_quality_report, log_returns, simple_returns

In [ ]:
prices = pd.Series(
    [100.0, 101.5, 100.8, 102.4],
    index=pd.to_datetime(["2026-01-02", "2026-01-05", "2026-01-06", "2026-01-07"]),
    name="example_price",
)

pd.DataFrame(
    {
        "price": prices,
        "simple_return": simple_returns(prices),
        "log_return": log_returns(prices),
    }
)

## Missing data and asynchronous calendars

Cross-market data often mixes Mexican and US holidays. Dropping every row with a missing observation can remove useful information and distort correlations. Forward fill is usually acceptable for last-traded prices, but not for all variables.

| Data type | Common treatment | Warning |
| --- | --- | --- |
| Adjusted equity prices | align to business calendar and forward fill | do not fill through long suspensions without review |
| Returns | compute after price alignment | do not forward-fill returns directly |
| Macroeconomic levels | align by release frequency | revisions and publication lags matter |
| Yield curve points | interpolate carefully | avoid artificial zero-volatility segments |

In [ ]:
from src.market_data_quality import align_to_business_calendar

aligned_prices = align_to_business_calendar(prices)
data_quality_report(aligned_prices)

## Data revisions

Macroeconomic data may be revised after initial publication. This creates a distinction between latest available data, first-release data, vintage data, and real-time data available on a historical date. A strategy tested using today's revised macro data may accidentally use information that was not available at the time of the decision {cite}`alfredVintageData`.

For a classroom dashboard, latest data can be acceptable if the limitation is stated. For a historical simulation, the release calendar and revision policy become part of the model design.

## Corporate actions

Equity returns should normally be computed from adjusted prices. Splits and dividends can create artificial price jumps if raw closing prices are used without adjustment. A split-adjusted or total-return-adjusted price series is often the correct modeling input for VaR, portfolio optimization, and volatility modeling.

When using `yfinance`, prefer adjusted price fields or use `auto_adjust=True` when appropriate. Always state which field was used.

## Robust outlier flags

Financial returns are heavy-tailed, so a large move is not automatically bad data. The goal is to flag observations for audit, not to erase real market stress.

The Hampel idea compares each observation with a rolling local median {cite}`hampel1974influence`:

$$
MAD_t = \text{median}(|X_{t-k} - M_t|,\dots,|X_{t+k}-M_t|).
$$

An observation is flagged when:

$$
|X_t - M_t| > n \times 1.4826 \times MAD_t.
$$

In [ ]:
from src.market_data_quality import hampel_outlier_flags

returns = log_returns(prices)
hampel_outlier_flags(returns, window=3, n_sigmas=3.0)

## Validation checklist

Before using a dataset in a model or dashboard, confirm:

```text
1. Are dates parsed correctly?
2. Are there duplicated rows?
3. Are expected columns present?
4. Are numeric columns numeric?
5. Are units documented?
6. Are missing dates expected?
7. Are missing values explained?
8. Are extreme values flagged?
9. Are prices adjusted or unadjusted?
10. Are time zones and calendars documented?
11. Are source and retrieval date stored?
12. Are transformations reproducible?
```

Validation does not guarantee truth, but it reduces avoidable errors.

## Assumptions log

Every dataset should include a short assumptions log.

| Field | Example |
| --- | --- |
| Dataset | `mx_market_panel.parquet` |
| Source | Banxico, INEGI, BMV, yfinance |
| Retrieval date | `YYYY-MM-DD` |
| Calendar | Mexican business days |
| Missing values | Forward-filled only for policy rate |
| Prices | Adjusted close for equities |
| Inflation | Monthly year-over-year CPI |
| Exchange rate | Daily close |
| Known limitations | Yahoo data used only for educational purposes |

A documented assumption is not automatically correct, but it is auditable.

## Modeling handoff

Data quality decisions affect later modules:

| Decision | Downstream effect |
| --- | --- |
| adjusted versus raw prices | return calculation, VaR, volatility, portfolio optimization |
| calendar alignment | correlation, covariance, beta, spread analysis |
| outlier treatment | volatility estimates, GARCH parameters, tail-risk metrics |
| macro release frequency | time series model selection and interpretation |
| currency and inflation units | fixed-income and real-return analysis |